# DEM Workflow (Notebook)

This notebook does one workflow: pick a bbox (default Norway), download Copernicus DEM tiles, and merge them into one DEM file.

## Inputs

- Set `bbox_text` to `None` to use the default Norway bbox.
- Or set `bbox_text` to a custom bbox string: `min_lon,min_lat,max_lon,max_lat`.

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv

from src.bbox import NORWAY_BBOX, parse_bbox
from src.dem import (
    create_temp_s3_credentials,
    delete_temp_s3_credentials,
    download_dem_products,
    get_access_token,
    merge_dems,
    s3_client_from_creds,
    save_bbox_json,
    search_cop_dem_products,
)

In [5]:
load_dotenv()

bbox_text = None
# Example custom bbox: bbox_text = "7.6,58.1,8.8,59.1"

base_dir = project_root
tiles_dir = base_dir / "dem_downloads"
output_dem = base_dir / "data/processed/dem_merged.tif"
output_bbox = base_dir / "data/processed/aoi_bbox.json"
max_products = None

bbox = parse_bbox(bbox_text) if bbox_text else NORWAY_BBOX
save_bbox_json(bbox, output_bbox)
print(f"Using bbox: {bbox}")
print(f"Tiles folder: {tiles_dir}")
print(f"Merged DEM path: {output_dem}")

Using bbox: (4.5, 57.8, 31.3, 71.3)
Tiles folder: /Users/selcukoner/Desktop/cassini/dem_downloads
Merged DEM path: /Users/selcukoner/Desktop/cassini/data/processed/dem_merged.tif


In [4]:
token = get_access_token()
creds = create_temp_s3_credentials(token)
access_id = creds.get("access_id")

try:
    s3_client = s3_client_from_creds(creds)
    products = search_cop_dem_products(bbox, token)
    print(f"Found {len(products)} DEM products in this bbox")

    tiles = download_dem_products(
        products=products,
        s3_client=s3_client,
        output_dir=tiles_dir,
        max_products=max_products,
    )
    print(f"Downloaded or reused {len(tiles)} DEM tiles")

    merged = merge_dems(tiles, output_dem)
    print(f"Merged DEM saved to: {merged}")
finally:
    if access_id:
        delete_temp_s3_credentials(access_id, token)

Found 200 DEM products in this bbox
Downloaded or reused 161 DEM tiles
Merged DEM saved to: data/processed/dem_merged.tif
